# 1. Import Modules

In [ ]:
import csv
import json
import requests
import yaml
from datetime import datetime

# 2. Authenticate to the API

In [ ]:
#Get the current date. This will be used to name the files created.
report_date = datetime.now().strftime("%m_%d_%Y")

#Safe loads the secrets file containing your username and password.
with open("../secrets.yml") as f:
    secrets = yaml.safe_load(f)

#Credentials to post for authentication
baseURL = 'https://archives.pratt.edu/staff/api'
user = secrets['username']
password = secrets['password']
repository = "2" #The Pratt Archives has only one repository which will always be "2"

#Sends the authentication request to the API
auth = requests.post(baseURL + '/users/' + user + '/login?password='+ password).json()

#If authentication fails, an error will be printed.
if 'session' not in auth:
    print("Error: authentication failed.")
    print(auth)
else:
    session = auth['session']
    headers = {'X-ArchivesSpace-Session': session,
            'Content_Type': 'application/json'}

    print('Authentication successful.')

# 3. Get Agents
There are a lot of agents!! This will take a wile (~20 minutes)

In [ ]:
#Agents exist in four different categories: 
agent_types = ['corporate_entities', 'families', 'people']

records = []

#Loop over each endpoint
for agent_type in agent_types:

    endpoint = f'/agents/{agent_type}?all_ids=true'

    #Get list of IDs
    ids = requests.get(baseURL + endpoint, headers=headers).json()
    
    print(f"{len(ids)} {agent_type} found. Getting information...")

    counter = 1

    #Iterate over IDs 
    for id in ids:
        output = requests.get(f"{baseURL}/agents/{agent_type}/{id}", headers=headers).json()
        records.append(output)
        
        if counter % 10 == 0: 
            print(f"Status: {counter}/{len(ids)}")
        
        counter += 1

#Data is dumped into a JSON file
with open(f'../output/agents_{report_date}.json', 'w', encoding='utf-8') as f:
    json.dump(records, f)

print(f"Completed.")

# 4. Create Column Headers for CSV file

In [ ]:
with open(f'../output/agents_{report_date}.json', 'r', encoding='utf-8') as json_file:
    json_data = json.load(json_file)

csv_fields = []

#Top-level keys from the JSON file
base_fieldnames = [
    "uri",
    "title",
    "agent_type",
    "publish",
    "is_linked_to_published_record",
    "is_repo_agent",
    "related_agents" #This is a boolean for if there are related agents
]

csv_fields.extend(base_fieldnames)

#Fields that are nested within the "display_name" dictionary.
nested_fieldnames = [
    "name_order",
    "sort_name",
    "prefix",
    "title",
    "primary_name",
    "rest_of_name",
    "family_name",
    "fuller_form",
    "location",
    "suffix",
    "dates",
    "rules",
    "source",
    "authority_id"
]

csv_fields.extend([f"display_name_{field}" for field in nested_fieldnames])

# Create columns based on the maximum number of linked agent roles
max_linked_agent_roles = max(len(r.get("linked_agent_roles", [])) for r in json_data)

for i in range(1, max_linked_agent_roles + 1):
    csv_fields.append(f"linked_agent_role_{i}")

print(f"Columns to be added to CSV: {json.dumps(csv_fields, indent=2)}")



# 5. Write JSON to CSV

In [ ]:
with open(f'../output/agents_{report_date}.json', 'r', encoding='utf-8') as json_file:
    json_data = json.load(json_file)

with open(f'../output/agents_{report_date}.csv', 'w', newline="", encoding='utf-8') as csv_file:
    csv_writer = csv.DictWriter(csv_file, fieldnames=csv_fields)
    csv_writer.writeheader()

    for entry in json_data:

        row = {}

        #Retrieves top-level fields.
        for b in base_fieldnames:
            if b == "related_agents":
                row[b] = bool(entry.get(b))
            else:
                row[b] = entry.get(b, "")

        #Retrieves data for fields that are nested within the "display_name" dictionary.
        display_name = entry.get("display_name") or {}

        for n in nested_fieldnames:
            row[f"display_name_{n}"] = display_name.get(n, "")

        # Retrieves extents.
        for i, role in enumerate(entry.get("linked_agent_roles", []), start=1):
            row[f"linked_agent_role_{i}"] = role
        
        csv_writer.writerow(row)

